# Marker diversity and sphericity filter inspection

**Role:** Analysis.

Check whether the sphericity filter rejects organoids that still contain a diverse set of fate markers. This notebook reads source graphs and geometry metadata, compares marker-diversity distributions for retained and rejected organoids, and lets you inspect one marker-colored mesh at a time using sphericity and diversity intervals. The browser includes both retained and rejected organoids. It does not load or train a prediction model, measure prediction errors, or apply a marker-diversity filter. Diversity here describes whole-organoid marker composition, not spatial neighborhood complexity or marker coexpression patterns.


In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/models/gnn.py").is_file())  # Locate the repository from the notebook’s working directory.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
ROOT = PROJECT_ROOT  # Repository root used to resolve data and result paths.


## Analysis settings

Choose the source dataset, sphericity filter threshold, and initial intervals for the mesh browser. Adjust the intervals interactively later; these choices do not modify source data or training settings.


In [ ]:
# Initial browser intervals
SPHERICITY_RANGE = None  # Inclusive Q interval; None starts with the full measured range.
DIVERSITY_RANGE = None  # Inclusive diversity interval within [0, 1]; None includes all scores.

# Marker display colors
DEFAULT_MARKER_COLORS = {
    "Lysozyme": "#2C75D2",  # Display color for Lysozyme.
    "Serotonin": "#F392E3",  # Display color for Serotonin.
    "Mucin 2": "#3EC1D9",  # Display color for Mucin 2.
    "Chroma": "#D852CB",  # Display color for Chroma.
    "Glucagon": "#983EE2",  # Display color for Glucagon.
    "LGR5": "#FFB431",  # Display color for LGR5.
    "AldoB": "#F16C6A",  # Display color for AldoB.
    "Agr2": "#359BD5",  # Display color for Agr2.
    "KI67": "#808080",  # Display color for KI67.
    "none": "#EBEBEB",  # Display color for none.
}  # Display colors for cell identities; not model inputs.

# Morphology filter
SPHERICITY_MAX = .92  # Maximum retained sphericity for the filter inspection.

# Input runs, data and destinations
data_dir = PROJECT_ROOT / "training_data" / "after_cleanup"  # Dataset directory supplying graphs and metadata.


## Imports

Load the packages and shared helpers used below. The repository root is defined once in the bootstrap cell, and the dataset directory is selected once in the settings cell.


In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from src.data.io import load_graph_dataset_from_dir
from src.data.metadata import load_aux_metadata_for_dir, attach_metadata_to_graphs, load_marker_names_from_dir
from src.data.filters import filter_graphs_by_sphericity, marker_diversity_scores
import ipywidgets as widgets
from IPython.display import display
from src.plotting.mesh_plots import project_marker_categories_for_graph, plot_categorical_mesh

print("Dataset:", data_dir)


## Load the source cohort

Read graphs and auxiliary metadata from the selected dataset. No training split or model is required for this inspection.

In [ ]:
if not data_dir.is_dir():
    raise FileNotFoundError(f"Dataset directory does not exist: {data_dir}. Update data_dir in the settings cell.")
graphs = load_graph_dataset_from_dir(str(data_dir))
if not graphs:
    raise ValueError(f"No usable organoid graphs found in {data_dir}. Check the dataset setting and NPZ files.")
print(f"Loaded {len(graphs)} organoids.")
print("Raw y shape:", tuple(graphs[0].y.shape))

meta = load_aux_metadata_for_dir(str(data_dir))
attach_metadata_to_graphs(graphs, meta)

source_graphs = graphs  # Keep the full cohort for interval-based inspection.

marker_names = load_marker_names_from_dir(data_dir)
if not marker_names:
    raise ValueError(f"Missing *_markers.json metadata in {data_dir}; restore marker names in feature-column order.")
if any(g.x.shape[1] != len(marker_names) for g in source_graphs):
    raise ValueError("Saved marker names do not match graph feature widths.")


## Inspect the sphericity filter

Keep organoids with isoperimetric quotient $Q=36\pi V^2/A^3$ below `SPHERICITY_MAX` (a sphere has $Q=1$). Print retained counts by dataset and timepoint. Rejected organoids also include those with missing or invalid surface area or volume, so rejection does not necessarily imply a spherical shape. This changes only the in-memory selection.


In [ ]:
graphs, rejected, sphericity_scores = filter_graphs_by_sphericity(
    source_graphs,
    max_sphericity=SPHERICITY_MAX,
    print_summary=True,
    return_rejected=True,
    return_scores=True,
)

## Compare fate diversity

For each organoid, calculate the positive-cell fraction $f_m$ for every marker and average $f_m/(f_m+0.01)$ across markers with equal weights. A marker present in 1% of cells contributes 0.5; absent markers contribute zero, and common markers approach one. The resulting score lies between zero and one and measures marker representation rather than spatial arrangement. Coexpressed markers are counted separately; unassigned cells are not an additional marker. Both groups use the same histogram bins and axes for comparison.


In [ ]:
scores_graphs = marker_diversity_scores(graphs)
scores_rejected = marker_diversity_scores(rejected)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharex=True, sharey=True)
for ax, scores, label, color in [
    (axes[0], scores_graphs, "Retained", "tab:blue"),
    (axes[1], scores_rejected, "Rejected (including invalid geometry)", "tab:orange"),
]:
    ax.hist(scores, bins=np.linspace(0, 1, 31), color=color)
    ax.set_title(f"{label}\n{len(scores):,} organoids")
    ax.set_xlabel("Marker diversity score")
    if len(scores) == 0:
        ax.text(0.5, 0.5, "No organoids in this group", ha="center", transform=ax.transAxes)
axes[0].set_ylabel("Number of organoids")
fig.suptitle(f"Marker diversity after sphericity filtering (Q < {SPHERICITY_MAX:g})")
fig.tight_layout()
plt.show()

inspection_table = pd.DataFrame({
    "organoid": [g.organoid_str for g in source_graphs],
    "sphericity": sphericity_scores,
    "diversity": marker_diversity_scores(source_graphs),
    "passes_sphericity_filter": np.isfinite(sphericity_scores) & (sphericity_scores < SPHERICITY_MAX),
    "num_cells": [len(g.x) for g in source_graphs],
}).set_index("organoid")


## Browse organoids by sphericity and marker diversity

Set both inclusive intervals, choose one of the matching organoids, and click **Display mesh**. The browser uses the full source cohort, including organoids rejected by the sphericity filter. The preview reports each organoid's scores and filter status. Missing or invalid sphericity cannot be matched to a numerical interval and is excluded. Changing a selection clears the previous mesh; meshes are loaded only when requested.

Colors use the first positive marker in `DEFAULT_MARKER_COLORS` order, while the diversity score counts every positive marker. Original mesh files referenced in auxiliary metadata must be available; a missing mesh is reported for the selected organoid.


In [ ]:
def complexity_candidates(table, sphericity_range, diversity_range):
    """Select finite scores inside both inclusive intervals."""
    for bounds in (sphericity_range, diversity_range):
        if len(bounds) != 2 or not np.isfinite(bounds).all() or bounds[0] > bounds[1]:
            raise ValueError("Intervals must contain two ordered finite bounds.")
    return table[
        np.isfinite(table.sphericity) & np.isfinite(table.diversity)
        & table.sphericity.between(*sphericity_range)
        & table.diversity.between(*diversity_range)
    ].sort_values(["diversity", "sphericity"], ascending=[False, True])


def build_complexity_browser(table, graph_lookup, marker_names, marker_colors, *,
                             sphericity_range=None, diversity_range=None):
    """Select a single source organoid without loading meshes until requested."""
    if table.empty or not table.index.is_unique:
        raise ValueError("The inspection table must contain uniquely identified organoids.")
    finite_q = table.loc[np.isfinite(table.sphericity), "sphericity"]
    q_min = min(0., float(finite_q.min())) if len(finite_q) else 0.
    q_max = max(1., float(finite_q.max())) if len(finite_q) else 1.
    sphericity_range = (q_min, q_max) if sphericity_range is None else tuple(sphericity_range)
    diversity_range = (0., 1.) if diversity_range is None else tuple(diversity_range)
    complexity_candidates(table, sphericity_range, diversity_range)  # Validate initial intervals.
    if not 0 <= diversity_range[0] <= diversity_range[1] <= 1:
        raise ValueError("DIVERSITY_RANGE must lie within [0, 1].")
    style = {"description_width": "initial"}
    q_control = widgets.FloatRangeSlider(
        value=sphericity_range, min=min(q_min, sphericity_range[0]), max=max(q_max, sphericity_range[1]),
        step=.001, description="Sphericity Q", readout_format=".3f", continuous_update=False,
        style=style, layout=widgets.Layout(width="95%"))
    diversity_control = widgets.FloatRangeSlider(
        value=diversity_range, min=0., max=1., step=.001, description="Marker diversity",
        readout_format=".3f", continuous_update=False, style=style, layout=widgets.Layout(width="95%"))
    organoid = widgets.Dropdown(options=[], description="Organoid", style=style,
                               layout=widgets.Layout(width="95%"))
    show_mesh = widgets.Button(description="Display mesh", button_style="primary")
    status = widgets.HTML()
    preview = widgets.HTML()
    mesh_output = widgets.Output()
    state = {"updating": False}

    def preview_selection(change=None):
        if state["updating"]:
            return
        mesh_output.outputs = ()
        # Replace one HTML table directly, without IPython output capture.
        preview.value = (
            table.loc[[organoid.value]].to_html(border=0, escape=True)
            if organoid.value is not None else ""
        )

    def update_options(change=None):
        eligible = complexity_candidates(table, q_control.value, diversity_control.value)
        prior = organoid.value
        options = [(f"Q {row.sphericity:.3f} | diversity {row.diversity:.3f} | {name}", name)
                   for name, row in eligible.iterrows()]
        state["updating"] = True
        try:
            organoid.options = options
            organoid.value = prior if prior in eligible.index else (options[0][1] if options else None)
            organoid.disabled = show_mesh.disabled = not bool(options)
        finally:
            state["updating"] = False
        n_missing = int((~np.isfinite(table.sphericity)).sum())
        status.value = (f"{len(eligible):,} / {len(table):,} organoids match both inclusive intervals. "
                        f"{n_missing:,} with missing/invalid Q excluded.")
        if not options:
            status.value += " No organoids match; widen the intervals."
        preview_selection()

    def render_selected(button=None):
        name = organoid.value
        if name is None:
            return
        with mesh_output:
            mesh_output.outputs = ()
            try:
                projection = project_marker_categories_for_graph(
                    graph_lookup[name], marker_names=marker_names, marker_colors=marker_colors,
                    missing_category="none")
                row = table.loc[name]
                figure = plot_categorical_mesh(
                    projection["mesh"], projection["mesh_categories"],
                    category_order=[m for m in marker_colors if m != "none" and m in marker_names],
                    category_colors=marker_colors, baseline_color=marker_colors.get("none", "lightgray"),
                    baseline_name="none", add_legend=True,
                    title=f"{name} | Q={row.sphericity:.3f} | diversity={row.diversity:.3f}")
                # Write one MIME representation directly; bypass automatic Plotly display.
                mesh_output.outputs = ({
                    "output_type": "display_data",
                    "data": {"application/vnd.plotly.v1+json": json.loads(figure.to_json())},
                    "metadata": {},
                },)
            except Exception as exc:
                print(f"Cannot display mesh for {name}: {type(exc).__name__}: {exc}")

    for control in (q_control, diversity_control):
        control.observe(update_options, names="value")
    organoid.observe(preview_selection, names="value")
    show_mesh.on_click(render_selected)
    update_options()
    browser = widgets.VBox([q_control, diversity_control, status, organoid, preview, show_mesh, mesh_output])
    return browser, dict(sphericity=q_control, diversity=diversity_control, organoid=organoid,
                         show_mesh=show_mesh, status=status, mesh_output=mesh_output, preview=preview)


complexity_browser, complexity_controls = build_complexity_browser(
    inspection_table, {g.organoid_str: g for g in source_graphs}, marker_names, DEFAULT_MARKER_COLORS,
    sphericity_range=SPHERICITY_RANGE, diversity_range=DIVERSITY_RANGE)
display(complexity_browser)
